# Домашнее задание 2: объяснения на всех модальностях

**Тема:** один и тот же набор идей — градиенты, пертурбации, суррогаты, контрфактуалы — на таблицах, изображениях, тексте, временных рядах, аудио, графах, рекомендациях и в мультимодальной модели.

**Дата выдачи:** 24.10 

**Дедлайн:** 14.11 23:59 МСК

**Баллы:** 10 базовых + 1 бонусный (максимум 11)

---

## О чём это ДЗ

На занятиях 04 и 06–07 мы разобрали методы атрибуции по модальностям. Мы видели, что любой метод задаётся четырьмя решениями:

1. **Единица объяснения** — признак, пиксель, суперпиксель, токен, окно ряда, ячейка спектрограммы, ребро графа, item в истории пользователя.
2. **Процесс** — что мы делаем с единицей: берём градиент, заменяем, удаляем, маскируем.
3. **Baseline** — на что заменяем: ноль, среднее, [MASK], тишину, размытие.
4. **Цель** — какое число объясняем: логит, вероятность, разность логитов, скор рекомендации, косинус. Частично сопряжено со структурой данных, на которой строится объяснение. 

В каждой части вашей задачей будет менять выбирать из 4 компонент смотреть, как они меняют ответ. Там, где это возможно, мы строим данные с **известной истиной** (синтетика) — чтобы проверять объяснения числами. В реальных условиях сюда налагается то, что найденное объяснение — суть гипотеза, а не причинно=следственная связь.

**Как сдавать:** заполненный ноутбук с выполненными ячейками. Ответы на вопросы — в ячейках «_Ваш ответ:_». Ответ без числа или графика, на который он опирается, оценивается в половину. Число или график без вывода — аналогично штрафуется. 

**Ресурсы:** всё считается на CPU ноутбука за ~15–20 минут. Модели скачиваются при первом запуске (ResNet-50 ~100 МБ, DistilBERT ~270 МБ, AST ~350 МБ, CLIP ~600 МБ). Не страдайте с Каггл, для этого есть другие предметы. 

## Часть 0. Сетап

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import io, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import spearmanr

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})
np.random.seed(42)
torch.manual_seed(42)
torch.set_num_threads(max(1, torch.get_num_threads()))
DEVICE = "cpu"  # всё ДЗ рассчитано на CPU; на GPU работает быстрее, но числа могут чуть отличаться
DATA_DIR = "./data"  # сюда скачаются Imagenette и MovieLens

---
## Часть 1. Таблицы: глобальные эффекты и взаимодействия (1,4 балла)

Веса, важности деревьев, SHAP и LIME на таблицах вы уже разобрали в Stepik-курсе (ДЗ-1). Здесь — то, чего там не было: **PDP, ICE, ALE и H-статистика**.

Данные — прокат велосипедов в Вашингтоне (Bike Sharing, 17 379 часов). Цель — число прокатов за час. Признаки `temp` (температура) и `feel_temp` (ощущаемая температура) коррелированы почти идеально — это нам и нужно.

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.inspection import partial_dependence, permutation_importance

bike = fetch_openml("Bike_Sharing_Demand", version=2, as_frame=True).frame
for col in ["season", "weather"]:
    bike[col] = bike[col].cat.codes
for col in ["holiday", "workingday"]:
    bike[col] = bike[col].astype(str).eq("True").astype(int)

X = bike.drop(columns="count").astype(float)
y = bike["count"].astype(float)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)

gbm = HistGradientBoostingRegressor(max_iter=300, random_state=0).fit(X_tr, y_tr)
print(f"R^2 на тесте: {gbm.score(X_te, y_te):.3f}")
print(f"corr(temp, feel_temp) = {np.corrcoef(X.temp, X.feel_temp)[0, 1]:.3f}")

S = X_te.sample(1000, random_state=0)  # выборка, на которой считаем все эффекты

### 1.1 PDP и ICE своими руками (0,4 балла)

Напоминание: для признака $j$ и значения $v$

$$\text{ICE}_i(v) = f(v, x_{i,-j}), \qquad \text{PDP}(v) = \frac1n\sum_i \text{ICE}_i(v).$$

**a) [0,2 балла]** Реализуйте `ice(model, X, feature, grid)` (возвращает матрицу `n × len(grid)`) и `pdp(...)` через неё. Сверьте `pdp` для `hour` на сетке `0..23` с `sklearn.inspection.partial_dependence` (передайте ту же сетку через `custom_values` и `method="brute"`) — максимальное расхождение должно быть ~0. *(Для бустинга sklearn по умолчанию берёт `method="recursion"`, который считает PD по весам листьев обучающей выборки, а не по `S`, — числа будут другими.)*

**b) [0,2 балла]** Постройте **центрированные** ICE-кривые для `hour` ($\text{ICE}_i(v) - \text{ICE}_i(0)$), раскрасив их по `workingday`, и поверх — PDP. Выведите средние центрированные ICE для рабочих и выходных дней в часы 8, 12, 17.

In [ ]:
def ice(model, X, feature, grid):
    # CODE PLEASE: для каждого v из grid подставьте feature=v во ВСЕ строки X и сделайте predict
    #              верните матрицу (n_objects, len(grid))
    pass

def pdp(model, X, feature, grid):
    # CODE PLEASE
    pass

hours = np.arange(24)
# CODE PLEASE: сверьте pdp(gbm, S, "hour", hours) с partial_dependence(..., custom_values={"hour": hours})

# CODE PLEASE: центрированные ICE (вычтите значение при hour=0), раскраска по S["workingday"], поверх PDP
# CODE PLEASE: таблица средних центрированных ICE для рабочих/выходных в часы 8, 12, 17

**Вопрос [без баллов, но ответ нужен для 1.4].** Что PDP «прячет» в этом признаке? Как бы выглядела PDP, если бы в рабочие дни пики были в 8 и 17, а в выходные — ровно в противофазе с той же амплитудой?

_Ваш ответ:_

### 1.2 ALE против PDP на коррелированных признаках (0,4 балла)

PDP подставляет значение $v$ во **все** объекты, включая те, где такой комбинации не бывает: `temp = 35°` при `feel_temp = 5°`. ALE (Apley & Zhu, 2020) считает **локальные разности** только внутри интервала, где объект реально лежит, и накапливает их:

$$\text{ALE}(v) = \sum_{k:\, z_k \le v} \frac{1}{n_k}\sum_{i:\, x_{ij}\in(z_{k-1}, z_k]} \big[f(z_k, x_{i,-j}) - f(z_{k-1}, x_{i,-j})\big] \; - \; \text{const}.$$

**a) [0,2 балла]** Реализуйте `ale(model, X, feature, n_bins=20)` с квантильными границами $z_k$; центрируйте результат так, чтобы среднее по объектам было 0. Верните `(границы, значения)`.

**b) [0,1 балла]** Постройте на одном графике центрированные PDP и ALE для `temp` и отдельно для `feel_temp`.

**c) [0,1 балла]** Посчитайте долю пар «объект × точка сетки» в PDP для `temp`, которые лежат **вне облака данных**: назовём точку $(v, \text{feel\_temp}_i)$ «вне», если $|v - \hat t(\text{feel\_temp}_i)| > 3\sigma$, где $\hat t$ — линейная регрессия `temp ~ feel_temp`, $\sigma$ — std её остатков.

In [ ]:
def ale(model, X, feature, n_bins=20):
    # CODE PLEASE: квантильные границы z (np.unique!), номер бина каждого объекта,
    #              в каждом бине средняя разность f(z_k) - f(z_{k-1}) по объектам этого бина,
    #              накопленная сумма, центрирование (взвешенное по числу объектов в бинах)
    pass

# CODE PLEASE: графики PDP vs ALE для temp и feel_temp

# CODE PLEASE: доля точек сетки PDP для temp, лежащих вне облака (temp, feel_temp)

**Вопрос [без баллов, ответ пригодится в 1.3].** Где кривые PDP и ALE расходятся сильнее всего и почему именно там? Какой метод вы бы показали заказчику с вопросом «как погода влияет на спрос»?

_Ваш ответ:_

### 1.3 Permutation importance и коррелированные признаки (0,3 балла)

**a) [0,1 балла]** Посчитайте permutation importance (`n_repeats=10`, R² на тесте) для всех признаков.

**b) [0,1 балла]** Реализуйте **групповую** перестановку: одной и той же перестановкой строк перемешайте сразу `temp` и `feel_temp`. Сравните падение R² от группы с суммой индивидуальных важностей двух признаков.

In [ ]:
# CODE PLEASE: permutation_importance на тесте, n_repeats=10

def group_pi(model, X, y, cols, n_repeats=10, seed=0):
    # CODE PLEASE: одна перестановка строк применяется ко всем cols сразу; среднее падение model.score
    pass

# CODE PLEASE: сравните group_pi(["temp", "feel_temp"]) с суммой индивидуальных важностей

**Вопрос [0,1 балла].** Почему групповая важность больше суммы индивидуальных? Если бы вы удалили `feel_temp` и переобучили модель — как изменилась бы важность `temp`? Что из этого следует для фразы «температура — 4-й по важности признак»?

_Ваш ответ:_

### 1.4 Насколько сильны взаимодействия: H-статистика Фридмана (0,3 балла)

$$H^2_{jk} = \frac{\sum_i \big[\text{PD}_{jk}(x_{ij}, x_{ik}) - \text{PD}_j(x_{ij}) - \text{PD}_k(x_{ik})\big]^2}{\sum_i \text{PD}^2_{jk}(x_{ij}, x_{ik})},$$

где все PD центрированы. $H^2 = 0$ — взаимодействия нет, $H^2 = 1$ — эффект пары целиком из взаимодействия.

**[0,2 балла]** Реализуйте `h_statistic(model, X, f1, f2)` (на подвыборке 200 объектов — это $O(n^2)$ предсказаний) и посчитайте для пар: (`hour`, `workingday`), (`hour`, `year`), (`hour`, `humidity`), (`temp`, `feel_temp`).

In [ ]:
def h_statistic(model, X, f1, f2):
    # CODE PLEASE: centered_pd(features) — для каждого объекта i подставьте его значения features
    #              во все объекты X и усредните predict; затем формула H^2
    pass

S200 = X_te.sample(200, random_state=1)
pairs = [("hour", "workingday"), ("hour", "year"), ("hour", "humidity"), ("temp", "feel_temp")]
# CODE PLEASE

**Вопрос [0,1 балла].** Согласуется ли ранжирование пар с тем, что вы видели на ICE в 1.1? Почему у `temp × feel_temp` H² маленькая, хотя признаки почти совпадают — что H-статистика вообще не умеет «видеть»?

_Ваш ответ:_

---
## Часть 2. Изображения (1,6 балла)

ResNet-50 (ImageNet) и картинки из Imagenette — 10 классов ImageNet. Imagenette скачается (~100 МБ).

In [ ]:
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.datasets import Imagenette
from captum.attr import IntegratedGradients

weights = ResNet50_Weights.IMAGENET1K_V2
resnet = resnet50(weights=weights).eval().to(DEVICE)
for p in resnet.parameters():
    p.requires_grad_(False)
preprocess = weights.transforms()
CATS = weights.meta["categories"]
MEAN = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
STD = torch.tensor([0.229, 0.224, 0.225])[:, None, None]

inette = Imagenette(DATA_DIR, split="val", size="160px", download=True)
IN_IDX = [0, 217, 482, 491, 497, 566, 569, 571, 574, 701]  # индексы классов Imagenette в ImageNet

rng = np.random.default_rng(0)
ids = rng.choice(len(inette), 40, replace=False)
IMGS = torch.stack([preprocess(inette[i][0]) for i in ids])
LABELS = torch.tensor([IN_IDX[inette[i][1]] for i in ids])
with torch.no_grad():
    PRED = resnet(IMGS).argmax(1)
print(f"accuracy на 40 картинках: {(PRED == LABELS).float().mean():.2f}")

def show(img_t):
    # денормализованная картинка для imshow
    return (img_t * STD + MEAN).permute(1, 2, 0).clamp(0, 1).numpy()

show_ids = [0, 1, 2, 3]   # картинки для визуализаций

### 2.1 Градиент и SmoothGrad (0,3 балла)

**a) [0,1 балла]** Реализуйте `grad_map(x, c)`: $\left|\partial f_c / \partial x\right|$, максимум по каналам → карта `224×224`.

**b) [0,1 балла]** Реализуйте `smoothgrad(x, c, n=25, sigma=0.15)`: среднее `grad_map` по $n$ зашумлённым копиям, $\sigma$ — доля от размаха значений $x$.

**c) [0,1 балла]** Нарисуйте для 4 картинок: изображение, градиент, SmoothGrad.

In [ ]:
def grad_map(x, c):
    # CODE PLEASE: x.requires_grad_, backward от логита класса c, |grad| и максимум по каналам
    pass

def smoothgrad(x, c, n=25, sigma=0.15, seed=0):
    # CODE PLEASE
    pass

# CODE PLEASE: сетка 3×4 — картинка / |grad| / SmoothGrad для IMGS[0:4] и их предсказанных классов

### 2.2 Grad-CAM своими руками: какой слой? (0,5 балла)

Grad-CAM: для слоя с активациями $A^k$ (канал $k$)

$$\alpha_k = \frac{1}{HW}\sum_{h,w}\frac{\partial f_c}{\partial A^k_{hw}}, \qquad L_c = \mathrm{ReLU}\Big(\sum_k \alpha_k A^k\Big),$$

затем апсемплинг до размера картинки.

**a) [0,2 балла]** Реализуйте `gradcam(x, c, layer)` через `register_forward_hook` и `register_full_backward_hook` (не забудьте удалять хуки). Обратите внимание: параметры сети заморожены, поэтому градиент придётся включить на входе (`x.requires_grad_(True)`), иначе backward-хук не сработает.

**b) [0,1 балла]** Постройте Grad-CAM для `layer2`, `layer3`, `layer4` на тех же 4 картинках. Выведите пространственное разрешение исходной (до апсемплинга) карты каждого слоя.

**c) [0,1 балла]** Для каждого слоя посчитайте долю «энергии» карты в центральном квадрате 112×112 (среднее по 40 картинкам) и число карт, которые после ReLU оказались целиком нулевыми. Доля в центре — грубый прокси «смотрит на объект»: в Imagenette объект обычно в центре.

In [ ]:
def gradcam(x, c, layer, return_raw=False):
    # CODE PLEASE: forward hook сохраняет активации, full backward hook — градиент по выходу слоя
    #              alpha_k = среднее градиента по пространству; cam = ReLU(sum_k alpha_k A_k)
    #              апсемплинг до 224×224, нормировка на максимум
    #              return_raw=True -> дополнительно верните форму карты до апсемплинга
    pass

layers = {"layer2": resnet.layer2, "layer3": resnet.layer3, "layer4": resnet.layer4}
# CODE PLEASE: сетка — картинка + Grad-CAM трёх слоёв для IMGS[0:4]
# CODE PLEASE: средняя доля энергии карты в центральном квадрате [56:168, 56:168] для каждого слоя
#              (осторожно: карта может оказаться целиком нулевой — посчитайте, сколько таких)

**Вопрос [0,1 балла].** Почему Grad-CAM классически берут с последнего свёрточного слоя? Что вы теряете и что выигрываете, опускаясь на `layer2`? Почему на промежуточных слоях карта может оказаться целиком нулевой?

_Ваш ответ:_

### 2.3 Integrated Gradients: полнота и выбор baseline (0,4 балла)

IG удовлетворяет аксиоме **полноты**: $\sum_i \text{IG}_i(x) = f(x) - f(x')$, где $x'$ — baseline. На практике интеграл считается суммой Римана, и ошибка видна в `convergence_delta`.

**a) [0,2 балла]** Для картинки `IMGS[0]` посчитайте IG (`captum`) с чёрным baseline при `n_steps ∈ {8, 32, 128, 512}` и покажите, как меняется `|delta|`. Проверьте полноту руками: сумма атрибуций vs $f(x)-f(x')$.

**b) [0,1 балла]** Сравните три baseline: **чёрный** (ноль в пространстве пикселей — внимание, после нормализации это не ноль тензора!), **размытый** (`GaussianBlur(kernel=51, sigma=20)`), **шумовой** (среднее IG по 8 случайным шумовым baseline). Постройте карты |IG| (сумма по каналам) и посчитайте попарные ранговые корреляции Спирмена между ними.

In [ ]:
from torchvision.transforms import GaussianBlur

ig = IntegratedGradients(resnet)
x, c = IMGS[:1], PRED[0].item()
black = ((torch.zeros(3, 224, 224) - MEAN) / STD)[None]   # чёрный в пиксельном пространстве

# CODE PLEASE: |delta| для n_steps in [8, 32, 128, 512]; сравните sum(attr) с f(x) - f(black)

# CODE PLEASE: IG с тремя baseline: black, GaussianBlur(51, sigma=20)(x), среднее по 8 шумовым
# CODE PLEASE: карты |IG|.sum(каналы), картинки и матрица ранговых корреляций Спирмена

**Вопрос [0,1 балла].** Объяснение IG — это всегда ответ на вопрос «чем $x$ отличается от $x'$». Сформулируйте, на какой вопрос отвечает каждая из трёх карт. Какой baseline выбрать для класса, где важен цвет (например, «golf ball» на зелёной траве)?

_Ваш ответ:_

### 2.4 Различает ли карта классы? (0,4 балла)

Хорошая атрибуция для класса $c_1$ должна отличаться от атрибуции для $c_2$, если классы про разное. Adebayo et al. (2018) и Selvaraju et al. (2017) показали, что «сырые» градиентные карты часто почти не зависят от класса и выглядят как детектор краёв.

**[0,3 балла]** Для всех 40 картинок возьмите top-1 и top-2 классы модели. Посчитайте корреляцию Спирмена между картами для top-1 и top-2 для трёх методов: |grad|, SmoothGrad (n=10), Grad-CAM (`layer4`). Постройте boxplot распределений корреляций.

In [ ]:
def class_corr(method):
    # CODE PLEASE: для каждой из 40 картинок — top-2 класса; карта для c1 и для c2; spearman по пикселям
    pass

methods = {
    "|grad|": grad_map,
    "SmoothGrad": lambda x, c: smoothgrad(x, c, n=10),
    "Grad-CAM (layer4)": lambda x, c: gradcam(x, c, resnet.layer4),
}
# CODE PLEASE: DataFrame корреляций по методам + boxplot

**Вопрос [0,1 балла].** Какой метод сильнее всего «зависит от класса»? Почему градиент по входу может почти не меняться при смене класса, хотя логиты разных классов — разные функции?

_Ваш ответ:_

---
## Часть 3. Текст (1,2 балла)

DistilBERT, дообученный на SST-2 (бинарная тональность). Объясняем **разность логитов** $f = \text{logit}_{pos} - \text{logit}_{neg}$: она не насыщается, как вероятность.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from captum.attr import LayerIntegratedGradients
from datasets import load_dataset

TXT = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
tok = AutoTokenizer.from_pretrained(TXT)
bert = AutoModelForSequenceClassification.from_pretrained(TXT, attn_implementation="eager").eval()

def f_text(input_ids):
    logits = bert(input_ids=input_ids).logits
    return logits[:, 1] - logits[:, 0]

def encode(s):
    ids = tok(s, return_tensors="pt").input_ids
    return ids, tok.convert_ids_to_tokens(ids[0])[1:-1]   # токены без [CLS]/[SEP]

sst = load_dataset("stanfordnlp/sst2", split="validation")
SENTS = [s for s in sst["sentence"] if 8 <= len(tok(s).input_ids) <= 24][:30]
print(len(SENTS), "предложений, пример:", SENTS[0])

### 3.1 Leave-one-out (0,3 балла)

**a) [0,2 балла]** Реализуйте `loo(sentence, mode)`: важность токена $i$ = $f(x) - f(x_{\setminus i})$, где $x_{\setminus i}$ получается заменой токена на `[MASK]` (`mode="mask"`) или его удалением (`mode="delete"`). `[CLS]` и `[SEP]` не трогаем.

**b) [0,1 балла]** Выведите обе версии для `"the movie was not good , but the actors were brilliant ."` и посчитайте корреляцию Спирмена между ними на 30 предложениях из `SENTS` (средняя по предложениям).

In [ ]:
@torch.no_grad()
def loo(sentence, mode="mask"):
    # CODE PLEASE: для каждого токена (кроме [CLS]/[SEP]) замените на [MASK] или удалите,
    #              важность = f(x) - f(x без токена); верните (tokens, scores)
    pass

demo = "the movie was not good , but the actors were brilliant ."
# CODE PLEASE: toks, s_mask = loo(demo, "mask"); s_del = ...; таблица LOO mask / LOO delete для demo
# CODE PLEASE: средняя корреляция Спирмена между двумя версиями на SENTS

### 3.2 Integrated Gradients по эмбеддингам (0,4 балла)

У текста нет непрерывного входа, поэтому IG считаем по **выходу слоя эмбеддингов** (`LayerIntegratedGradients` по `bert.distilbert.embeddings`), а атрибуцию токена берём как сумму по размерности эмбеддинга.

**a) [0,2 балла]** Baseline — последовательность, где все токены, кроме `[CLS]`/`[SEP]`, заменены на `[PAD]`. Для `demo` постройте график `|delta|` от `n_steps ∈ {10, 25, 50, 100, 200}`. Какое минимальное число шагов даёт `|delta| < 0.1`?

**b) [0,1 балла]** Сравните атрибуции IG с baseline `[PAD]` и `[MASK]` (по 200 шагов) на `demo`: выведите таблицу токен → IG(PAD), IG(MASK), LOO(mask).

In [ ]:
lig = LayerIntegratedGradients(f_text, bert.distilbert.embeddings)

def ig_text(sentence, base_token, n_steps=200):
    # CODE PLEASE: baseline = ids, где все токены кроме [CLS]/[SEP] заменены на base_token
    #              верните (tokens, атрибуция по токенам (сумма по эмбеддингу), |delta|)
    pass

# CODE PLEASE: |delta| от n_steps для demo с [PAD]-baseline (лог-шкала)
# CODE PLEASE: таблица IG [PAD] / IG [MASK] / LOO mask для demo

**Вопрос [0,1 балла].** Как меняется вклад слова `not` при смене baseline? Почему baseline `[MASK]` ближе по смыслу к LOO-mask, чем `[PAD]`?

_Ваш ответ:_

### 3.3 Внимание как объяснение? (0,3 балла)

**[0,2 балла]** Для 30 предложений из `SENTS` посчитайте три атрибуции: LOO (mask), IG ([MASK]-baseline, 100 шагов, по модулю) и **внимание** — среднее по головам последнего слоя внимание от `[CLS]` к токенам. Посчитайте средние попарные корреляции Спирмена между тремя методами (по |LOO|, |IG|, attention).

In [ ]:
@torch.no_grad()
def attention_cls(sentence):
    # CODE PLEASE: output_attentions=True, последний слой, среднее по головам, строка [CLS] -> токены
    pass

# CODE PLEASE: для каждого предложения из SENTS посчитайте |LOO|, |IG| (MASK, 100 шагов), attention
#              и попарные spearman; выведите средние

**Вопрос [0,1 балла].** Согласуется ли внимание с методами, которые реально вмешиваются во вход? Что именно измеряет внимание `[CLS]` последнего слоя, и почему это не «важность для предсказания» (вспомните residual stream и то, что после внимания стоят ещё MLP и классификатор)?

_Ваш ответ:_

### 3.4 Отрицание (0,2 балла)

**[0,1 балла]** Посчитайте LOO (mask) для четырёх предложений: `"the film is good ."`, `"the film is not good ."`, `"the film is bad ."`, `"the film is not bad ."`. Выведите $f(x)$ и вклад каждого токена.

In [ ]:
# CODE PLEASE: f(x) и LOO (mask) по токенам для четырёх предложений

**Вопрос [0,1 балла].** Получилось ли у `not` одно и то же по знаку значение в «not good» и «not bad»? Какое свойство аддитивных атрибуций (LOO, IG, SHAP) здесь мешает: можно ли одним числом на токен описать функцию «not *инвертирует* следующее слово»?

_Ваш ответ:_

---
## Часть 4. Временные ряды: синтетика с известной истиной (1,2 балла)

Чтобы проверять объяснение числом, построим задачу, где **мы знаем ответ**:

- каждый ряд — синусоида со случайной частотой, фазой и **смещением уровня** + шум;
- в половине рядов (класс 1) есть **гауссов «горб»** в случайном месте — именно он определяет класс;
- в 50% рядов **обоих классов** есть «глитч» — острый выброс, который классу не соответствует (дистрактор).

Классификатор — маленькая 1D-CNN; обучение ~30 секунд.

In [ ]:
L = 200

def make_series(n, seed=0):
    rng = np.random.default_rng(seed)
    t = np.arange(L)
    X = np.zeros((n, L), dtype=np.float32)
    y = rng.integers(0, 2, n)
    bump_pos = np.full(n, -1)
    for i in range(n):
        f, ph, lvl = rng.uniform(0.01, 0.05), rng.uniform(0, 2 * np.pi), rng.uniform(-2, 2)
        X[i] = lvl + 0.5 * np.sin(2 * np.pi * f * t + ph) + 0.3 * rng.standard_normal(L)
        if y[i] == 1:
            bump_pos[i] = rng.integers(20, L - 20)
            X[i] += 1.5 * np.exp(-0.5 * ((t - bump_pos[i]) / 4) ** 2)
        if rng.random() < 0.5:
            g = rng.integers(5, L - 5)
            X[i, g] += rng.choice([-3, 3])
    return torch.tensor(X)[:, None, :], torch.tensor(y), bump_pos

Xs_tr, ys_tr, _ = make_series(4000, seed=0)
Xs_te, ys_te, bump_te = make_series(1000, seed=1)

class CNN1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Conv1d(1, 16, 9, padding=4), nn.ReLU(),
                                 nn.Conv1d(16, 32, 9, padding=4), nn.ReLU(),
                                 nn.Conv1d(32, 32, 9, padding=4), nn.ReLU())
        self.head = nn.Linear(32, 2)
    def forward(self, x):
        x = x - x.mean(-1, keepdim=True)            # модель видит ряд без уровня
        return self.head(self.net(x).amax(-1))      # global max pool

torch.manual_seed(0)
cnn = CNN1D()
opt = torch.optim.Adam(cnn.parameters(), lr=3e-3)
for epoch in range(8):
    perm = torch.randperm(len(Xs_tr))
    for b in range(0, len(perm), 64):
        idx = perm[b:b + 64]
        loss = F.cross_entropy(cnn(Xs_tr[idx]), ys_tr[idx])
        opt.zero_grad(); loss.backward(); opt.step()
cnn.eval()
with torch.no_grad():
    acc = (cnn(Xs_te).argmax(1) == ys_te).float().mean().item()
print(f"accuracy на тесте: {acc:.3f}")

# объясняемые объекты: верно классифицированные ряды класса 1
with torch.no_grad():
    ok = (cnn(Xs_te).argmax(1) == ys_te) & (ys_te == 1)
EXPL = torch.where(ok)[0][:200]
print("рядов для оценки:", len(EXPL))

### 4.1 Gradient × Input (0,2 балла)

**[0,2 балла]** Реализуйте `grad_x_input(x)` для логита класса 1 и нарисуйте для 3 рядов из `EXPL`: сам ряд (отметьте истинное положение горба `bump_te`) и атрибуцию.

In [ ]:
def grad_x_input(x):
    # CODE PLEASE: атрибуция = grad(logit_1) * x, форма (batch, L)
    pass

# CODE PLEASE: 3 ряда из EXPL — ряд с отметкой bump_te и атрибуция под ним

### 4.2 Occlusion окнами: на что заменять? (0,4 балла)

**[0,4 балла]** Реализуйте `occlusion(x, width=10, stride=5, fill=...)`: окно скользит по ряду, внутри окна значения заменяются, атрибуция точки = среднее падение логита класса 1 по всем окнам, её накрывшим. Три варианта `fill`:

- `"zero"` — нули;
- `"mean"` — среднее значение **этого ряда**;
- `"interp"` — линейная интерполяция между значениями на краях окна.

Нарисуйте три атрибуции для тех же 3 рядов.

In [ ]:
@torch.no_grad()
def occlusion(x, width=10, stride=5, fill="mean"):
    # CODE PLEASE: окна [s, s+width) с шагом stride; замена по fill; падение логита класса 1;
    #              атрибуция точки = среднее падение по окнам, которые её накрывают
    pass

FILLS = ["zero", "mean", "interp"]
# CODE PLEASE: сетка 4×3 — ряд + три варианта occlusion для EXPL[:3]

### 4.3 Проверка числом: попадает ли атрибуция в горб? (0,4 балла)

Две метрики на рядах из `EXPL`:

- **pointing game** — доля рядов, где argmax атрибуции попал в окно $[\text{bump} - 8,\ \text{bump} + 8]$;
- **mass in window** — доля **положительной** атрибуции, приходящаяся на это окно (окно занимает 17/200 ≈ 8.5% ряда — это уровень случайного угадывания).

**[0,3 балла]** Посчитайте обе метрики для grad×input, трёх вариантов occlusion и случайной атрибуции. Отдельно посчитайте **долю рядов, где argmax попал в глитч** (±3 точки) — среди рядов, где глитч есть.

In [ ]:
def find_glitch(x):
    # дано: индекс глитча (или -1), если он есть в ряду
    d = np.abs(x - np.convolve(x, np.ones(9) / 9, mode="same"))
    j = int(np.argmax(d))
    return j if d[j] > 2.0 else -1

def evaluate_ts(attr_fn, n=200, seed=0):
    # CODE PLEASE: для рядов EXPL[:n]: pointing game, mass in window (по положительной части),
    #              доля argmax в глитче (±3) среди рядов с глитчем вне окна горба
    pass

ts_methods = {
    "grad×input": lambda x, r: grad_x_input(x)[0].numpy(),
    # CODE PLEASE: три occlusion и случайная атрибуция r.random(L)
}
# CODE PLEASE: таблица метрик

**Вопрос [0,1 балла + 0,2 балла за пункт 4.4 ниже].** Какой `fill` лучше и почему `"zero"` может вести себя плохо, хотя модель вычитает среднее ряда (подумайте, что происходит с рядом, когда вы зануляете окно в ряду с уровнем 2)? Притягивает ли глитч argmax атрибуции — и о чём это говорит: о методе или о модели? Как проверить, что модель действительно игнорирует глитч?

**4.4 [0,2 балла]** Сформулируйте, какой из методов вы бы использовали для объяснения модели, работающей на реальных сенсорных данных, и какой `fill` выбрали бы — с опорой на свои числа.

_Ваш ответ:_

---
## Часть 5. Аудио (1 балл)

Audio Spectrogram Transformer (AST), обученный на AudioSet (527 классов, multi-label). Его вход — log-mel спектрограмма `1024 кадра (10.24 с) × 128 мел-полос`. Мы снова строим вход **с известной истиной**: первые ~4 секунды — речь (LibriSpeech), с 6-й по 9-ю секунду — тон 1 кГц.

In [ ]:
import soundfile as sf
from datasets import Audio
from transformers import ASTFeatureExtractor, ASTForAudioClassification

AST = "MIT/ast-finetuned-audioset-10-10-0.4593"
fe = ASTFeatureExtractor.from_pretrained(AST)
ast = ASTForAudioClassification.from_pretrained(AST).eval()
ID2L = ast.config.id2label
L2ID = {v: k for k, v in ID2L.items()}

libri = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
libri = libri.cast_column("audio", Audio(decode=False))
speech, SR = sf.read(io.BytesIO(libri[0]["audio"]["bytes"]))
assert SR == 16000

clip = np.zeros(int(SR * 10.0))
sp = speech[: 4 * SR]
clip[: len(sp)] = 0.5 * sp / np.abs(sp).max()
t = np.arange(3 * SR) / SR
clip[6 * SR: 9 * SR] += 0.3 * np.sin(2 * np.pi * 1000 * t)
clip += 0.003 * np.random.default_rng(0).standard_normal(len(clip))

def ast_feats(wave):
    return fe(wave, sampling_rate=SR, return_tensors="pt").input_values   # (1, 1024, 128)

FEATS = ast_feats(clip)
SILENCE = ast_feats(np.zeros_like(clip))   # спектрограмма тишины после той же нормализации
FPS = 100   # кадров в секунду (шаг 10 мс)

### 5.1 Что слышит модель (0,2 балла)

**[0,2 балла]** Выведите топ-8 классов (sigmoid, т.к. multi-label) для `FEATS`. Нарисуйте спектрограмму (`FEATS[0].T`, время по X). Найдите классы, отвечающие за речь и за тон, и сохраните их индексы в `CLS_SPEECH`, `CLS_TONE`.

In [ ]:
# CODE PLEASE: LOGITS = ast(FEATS).logits[0]; топ-8 по sigmoid
# CODE PLEASE: CLS_SPEECH, CLS_TONE = ...   (найдите подходящие названия в топе)
# CODE PLEASE: спектрограмма FEATS[0].T (время по X, extent в секундах)

### 5.2 Окклюзия в частотно-временной плоскости (0,4 балла)

**[0,4 балла]** Разбейте спектрограмму на патчи `64 кадра × 16 мел-полос` (0.64 с × 1/8 диапазона), заменяйте патч **соответствующим куском `SILENCE`** и записывайте падение логита для `CLS_SPEECH` и `CLS_TONE`. Получите две карты `16 × 8` и нарисуйте их поверх спектрограммы (или рядом). Считайте батчами по 16.

In [ ]:
PT, PF = 64, 16

@torch.no_grad()
def tf_occlusion(feats, classes, fill, pt=PT, pf=PF, bs=16):
    # CODE PLEASE: патчи pt×pf заменяются на fill; падение логитов classes; карты (T/pt, F/pf, len(classes))
    pass

# CODE PLEASE: OCC = tf_occlusion(FEATS, [CLS_SPEECH, CLS_TONE], SILENCE) и две тепловые карты

### 5.3 Количественно (0,2 балла)

**[0,2 балла]** Для каждого класса посчитайте долю **положительной** атрибуции: (а) во временном интервале речи 0–4 с; (б) в интервале тона 6–9 с; (в) в мел-полосах, где лежит тон 1 кГц (найдите их по разности `FEATS` с тоном и без). Затем повторите окклюзию **только по времени** (патчи `64 × 128`, вся частотная ось) и сравните долю (а)/(б).

In [ ]:
t_sec = np.arange(OCC.shape[0]) * PT / FPS
# CODE PLEASE: найдите мел-полосы тона: разность ast_feats(clip) и ast_feats(clip без тона) в кадрах 600:900
# CODE PLEASE: доли положительной атрибуции во времени речи / тона / полосах тона для обоих классов
# CODE PLEASE: окклюзия только по времени (pf=128) и те же доли

**Вопрос [0,2 балла].** Объяснение «Beep» локализовалось и во времени, и по частоте? А «Speech»? Почему у Speech заметная доля атрибуции может приходиться на участок 6–9 с, где речи нет (подумайте, как «Beep» и «Speech» конкурируют в модели и что делает с картинкой вырезанный прямоугольник тишины на фоне шума)? Что именно мы предполагаем, когда выбираем «тишину» в качестве baseline для аудио?

_Ваш ответ:_

---
## Часть 6. Графы: BA-Shapes (1,2 балла)

Классический бенчмарк GNNExplainer (Ying et al., 2019): граф Барабаши–Альберт из 300 узлов, к которому прикреплены 80 «домиков» (5 узлов, квадрат + крыша) и добавлен 10% случайных рёбер. Метка узла — его **роль**: 0 — база, 1 — низ домика, 2 — середина, 3 — крыша. Признаков у узлов нет (все единицы), поэтому классифицировать можно **только по структуре** — и правильное объяснение для узла домика по построению = **рёбра его домика**.

GCN написан на чистом PyTorch (плотная матрица смежности), чтобы веса рёбер можно было дифференцировать и маскировать.

In [ ]:
import networkx as nx
from sklearn.metrics import roc_auc_score

def ba_shapes(n_base=300, n_houses=80, m=5, seed=0):
    rng = np.random.default_rng(seed)
    G = nx.barabasi_albert_graph(n_base, m, seed=seed)
    labels, motif = [0] * n_base, set()
    for _ in range(n_houses):
        s = G.number_of_nodes()
        G.add_nodes_from(range(s, s + 5))
        e = [(s, s + 1), (s + 1, s + 2), (s + 2, s + 3), (s + 3, s), (s, s + 4), (s + 1, s + 4)]
        G.add_edges_from(e); motif |= {tuple(sorted(x)) for x in e}
        labels += [1, 1, 2, 2, 3]
        G.add_edge(int(rng.integers(n_base)), s)
    n = G.number_of_nodes()
    for _ in range(int(0.1 * G.number_of_edges())):
        a, b = rng.integers(n, size=2)
        if a != b:
            G.add_edge(int(a), int(b))
    return G, np.array(labels), motif

GRAPH, YG, MOTIF = ba_shapes()
NN = GRAPH.number_of_nodes()
EDGES = np.array([tuple(sorted(e)) for e in GRAPH.edges()])
EI = torch.tensor(EDGES.T, dtype=torch.long)
IS_MOTIF = np.array([tuple(e) in MOTIF for e in EDGES])
XG = torch.ones(NN, 1)
print(f"{NN} узлов, {len(EDGES)} рёбер, из них в домиках: {IS_MOTIF.sum()}")

class GCN(nn.Module):
    def __init__(self, h=32, c=4):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(1, h), nn.Linear(h, h), nn.Linear(h, h)])
        self.out = nn.Linear(3 * h, c)
    def forward(self, x, edge_weight=None):
        if edge_weight is None:
            edge_weight = torch.ones(EI.shape[1])
        A = torch.zeros(NN, NN)
        A = A.index_put((EI[0], EI[1]), edge_weight, accumulate=True)
        A = A.index_put((EI[1], EI[0]), edge_weight, accumulate=True)
        A = A + torch.eye(NN)
        d = A.sum(1).pow(-0.5)
        A = d[:, None] * A * d[None]
        hs = []
        for layer in self.layers:
            x = F.relu(layer(A @ x)); hs.append(x)
        return self.out(torch.cat(hs, 1))

torch.manual_seed(0)
gcn = GCN()
yg = torch.tensor(YG)
perm = np.random.default_rng(0).permutation(NN)
tr_idx, te_idx = torch.tensor(perm[: int(0.8 * NN)]), torch.tensor(perm[int(0.8 * NN):])
opt = torch.optim.Adam(gcn.parameters(), lr=0.01, weight_decay=5e-4)
for epoch in range(1000):
    loss = F.cross_entropy(gcn(XG)[tr_idx], yg[tr_idx])
    opt.zero_grad(); loss.backward(); opt.step()
gcn.eval()
for p in gcn.parameters():
    p.requires_grad_(False)
with torch.no_grad():
    PRED_G = gcn(XG).argmax(1)
print(f"accuracy: train {(PRED_G[tr_idx] == yg[tr_idx]).float().mean():.2f}, test {(PRED_G[te_idx] == yg[te_idx]).float().mean():.2f}")

def khop_edge_mask(v, k=3):
    # рёбра внутри k-окрестности узла v — только они могут влиять на его предсказание в 3-слойной GCN
    nodes = set(nx.single_source_shortest_path_length(GRAPH, v, cutoff=k))
    return np.array([a in nodes and b in nodes for a, b in EDGES])

rng = np.random.default_rng(0)
TARGETS = rng.choice(np.where((YG > 0) & (PRED_G.numpy() == YG))[0], 30, replace=False)

### 6.1 Градиент по весам рёбер (0,3 балла)

**[0,3 балла]** Реализуйте `edge_saliency(v, c)`: $\left|\partial\, \text{logit}_c(v) / \partial w_e\right|$ для всех рёбер (веса рёбер — входной тензор единиц с `requires_grad`). Для узла `TARGETS[0]` нарисуйте его 3-окрестность, раскрасив рёбра по атрибуции, а рёбра домика выделите толщиной.

In [ ]:
def edge_saliency(v, c):
    # CODE PLEASE: w = torch.ones(число рёбер, requires_grad=True); gcn(XG, w)[v, c].backward(); |grad|
    pass

def draw_explanation(v, scores, title):
    # дано: рисует 3-окрестность v, цвет ребра = scores, толстые рёбра = домик
    sub = khop_edge_mask(v)
    H = nx.Graph(); H.add_edges_from(EDGES[sub])
    pos = nx.spring_layout(H, seed=0)
    sc = scores[sub] / (scores[sub].max() + 1e-12)
    nx.draw_networkx_nodes(H, pos, node_size=40, node_color=["red" if n == v else "lightgray" for n in H.nodes()])
    nx.draw_networkx_edges(H, pos, edgelist=[tuple(e) for e in EDGES[sub]], edge_color=sc, edge_cmap=plt.cm.viridis,
                           width=[3.5 if m else 0.8 for m in IS_MOTIF[sub]])
    plt.title(title); plt.axis("off")

# CODE PLEASE: draw_explanation для TARGETS[0]

### 6.2 GNNExplainer своими руками (0,4 балла)

GNNExplainer учит мягкую маску рёбер $M = \sigma(\theta)$ в окрестности узла, максимизируя вероятность предсказанного класса при маскированном графе и штрафуя размер маски и её «неопределённость»:

$$\min_\theta\; -\log p_c(v \mid M \odot A) \;+\; \lambda_{size}\sum_e M_e \;+\; \lambda_{ent}\,\overline{H(M_e)}.$$

**[0,4 балла]** Реализуйте `gnn_explainer(v, c, epochs=200, lr=0.01, size=0.005, ent=1.0)`. Маска вне 3-окрестности — 0. Инициализация $\theta \sim \mathcal N(0, 0.1)$ с фиксированным seed. Нарисуйте результат для `TARGETS[0]`.

In [ ]:
def gnn_explainer(v, c, epochs=200, lr=0.01, size=0.005, ent=1.0, seed=0):
    # CODE PLEASE: theta ~ N(0, 0.1) (torch.Generator с seed), маска M = sigmoid(theta) * (рёбра 3-окрестности)
    #              loss = -log_softmax(gcn(XG, M)[v])[c] + size * sum(M) + ent * средняя бинарная энтропия
    #              Adam по theta; верните M
    pass

# CODE PLEASE: draw_explanation для TARGETS[0] с маской GNNExplainer

### 6.3 AUC против истины и чувствительность к гиперпараметрам (0,3 балла)

**a) [0,2 балла]** Для узлов из `TARGETS` посчитайте ROC-AUC атрибуции рёбер 3-окрестности против `IS_MOTIF` (пропуская узлы, где все рёбра окрестности одного типа): для градиента, GNNExplainer с параметрами по умолчанию и случайной атрибуции.

**b) [0,1 балла]** Повторите GNNExplainer для сетки: `(lr, size, ent)` ∈ {(0.01, 0.005, 1.0), (0.1, 0.005, 1.0), (0.1, 0.0, 0.0), (0.1, 0.05, 1.0), (0.03, 0.005, 0.1)} на тех же узлах.

In [ ]:
def auc_over_targets(score_fn):
    # CODE PLEASE: для v из TARGETS — roc_auc_score(IS_MOTIF[sub], score_fn(v, YG[v])[sub]),
    #              пропуская окрестности с одним типом рёбер; верните (средний AUC, число узлов)
    pass

# CODE PLEASE: таблица AUC для градиента, GNNExplainer по умолчанию и random
# CODE PLEASE: AUC GNNExplainer по сетке гиперпараметров

**Вопрос [0,2 балла].** Что оказалось лучше: градиент или GNNExplainer? Насколько разброс по гиперпараметрам сопоставим с разницей между методами? Почему конфигурация `size=0, ent=0` близка к случайной (что оптимизирует маска, если ничто не штрафует её за «включённость»)? Как бы вы выбирали гиперпараметры объяснителя на **реальном** графе, где истины нет?

_Ваш ответ:_

---
## Часть 7. Рекомендации: «потому что вы смотрели X» (1,2 балла)

MovieLens-100k (943 пользователя, 1 682 фильма). Модель — матричная факторизация, обученная ALS. Для **фиксированных** векторов фильмов $V$ вектор пользователя — решение гребневой регрессии по его истории $H$:

$$u = (V_H^\top V_H + \lambda I)^{-1} V_H^\top (r_H - \mu) = W\, V_H^\top (r_H - \mu),$$

поэтому скор фильма $j$ раскладывается **точно**:

$$s_j = v_j^\top u = \sum_{i\in H} \underbrace{(r_i - \mu)\, v_j^\top W v_i}_{c_i}.$$

Это «объяснение по построению» (как веса линейной модели). Но $W$ сама зависит от $H$: если фильм $i$ убрать из истории, изменятся вклады **всех** остальных фильмов.

In [ ]:
import os, zipfile, urllib.request

ml_dir = os.path.join(DATA_DIR, "ml-100k")
if not os.path.exists(ml_dir):
    os.makedirs(DATA_DIR, exist_ok=True)
    zpath = os.path.join(DATA_DIR, "ml-100k.zip")
    urllib.request.urlretrieve("https://files.grouplens.org/datasets/movielens/ml-100k.zip", zpath)
    zipfile.ZipFile(zpath).extractall(DATA_DIR)

ratings = pd.read_csv(os.path.join(ml_dir, "u.data"), sep="\t", names=["u", "i", "r", "t"])
items = pd.read_csv(os.path.join(ml_dir, "u.item"), sep="|", encoding="latin-1", header=None).iloc[:, :2]
TITLE = dict(zip(items[0] - 1, items[1]))
NU, NI = ratings.u.max(), ratings.i.max()
R = np.zeros((NU, NI)); R[ratings.u - 1, ratings.i - 1] = ratings.r
MASK = R > 0
MU = ratings.r.mean()
K, LAM = 20, 5.0

rng = np.random.default_rng(0)
U = rng.normal(0, 0.1, (NU, K)); V = rng.normal(0, 0.1, (NI, K))
for it in range(15):
    for u in range(NU):
        h = MASK[u]; U[u] = np.linalg.solve(V[h].T @ V[h] + LAM * np.eye(K), V[h].T @ (R[u, h] - MU))
    for i in range(NI):
        h = MASK[:, i]
        if h.any():
            V[i] = np.linalg.solve(U[h].T @ U[h] + LAM * np.eye(K), U[h].T @ (R[h, i] - MU))
print(f"train RMSE: {np.sqrt((((U @ V.T + MU) - R)[MASK] ** 2).mean()):.3f}")

def user_vector(hist, ratings_hist):
    W = np.linalg.inv(V[hist].T @ V[hist] + LAM * np.eye(K))
    return W @ V[hist].T @ (ratings_hist - MU), W

def top_n(hist, ratings_hist, n=10):
    u, _ = user_vector(hist, ratings_hist)
    s = V @ u; s[hist] = -np.inf
    return np.argsort(-s)[:n], s

# пользователь с короткой историей — объяснение должно помещаться в голову
USER = int(np.where((MASK.sum(1) >= 25) & (MASK.sum(1) <= 35))[0][0])
HIST = np.where(MASK[USER])[0]
RH = R[USER, HIST]
TOP, SCORES = top_n(HIST, RH)
J = TOP[0]
print(f"пользователь {USER}, история {len(HIST)} фильмов; рекомендация №1: {TITLE[J]}")

### 7.1 Точная декомпозиция скора (0,4 балла)

**a) [0,3 балла]** Реализуйте `contributions(hist, r_hist, j)` → вектор $c_i$ и проверьте, что $\sum_i c_i = s_j$ с точностью до float.

**b) [0,1 балла]** Выведите топ-5 фильмов истории по $c_i$ с их оценками.

In [ ]:
def contributions(hist, r_hist, j):
    # CODE PLEASE: c_i = (r_i - MU) * v_j^T W v_i
    pass

# CODE PLEASE: проверка суммы и топ-5 вкладов

### 7.2 Контрфактуальный вклад: удалить из истории (0,3 балла)

**[0,3 балла]** Для каждого фильма истории посчитайте $\Delta_i = s_j(H) - s_j(H\setminus\{i\})$ (пересчитав $u$ без фильма). Посчитайте корреляцию Спирмена между $c_i$ и $\Delta_i$, нарисуйте scatter, найдите фильмы, у которых **знаки** $c_i$ и $\Delta_i$ различаются.

In [ ]:
def loo_effect(hist, r_hist, j):
    # CODE PLEASE: Δ_i = s_j(H) - s_j(H без i), где u пересчитывается через user_vector
    pass

# CODE PLEASE: Spearman(c, Δ), scatter, фильмы с разными знаками c_i и Δ_i

### 7.3 Минимальное контрфактуальное объяснение (0,3 балла)

«Если бы вы не смотрели {X, Y}, мы бы не порекомендовали J». Найдите **жадно** минимальный набор фильмов истории, после удаления которого $J$ выпадает из топ-10:

- стратегия A: удалять по убыванию $c_i$ (посчитанных один раз на полной истории);
- стратегия B: на каждом шаге пересчитывать $\Delta_i$ на текущей истории и удалять фильм с максимальным $\Delta_i$.

**[0,3 балла]** Реализуйте обе стратегии, выведите размер найденного набора и сами фильмы.

In [ ]:
def drops_out(hist, r_hist, j, n=10):
    # дано
    return j not in top_n(hist, r_hist, n)[0]

def greedy_static(hist, r_hist, j, max_k=15):
    # CODE PLEASE: удаляем фильмы в порядке убывания c_i (один раз посчитанных), пока j не выпадет из топ-10
    pass

def greedy_adaptive(hist, r_hist, j, max_k=15):
    # CODE PLEASE: на каждом шаге пересчитываем loo_effect и удаляем argmax
    pass

# CODE PLEASE: сравните размеры наборов

**Вопрос [0,2 балла].** Какое объяснение показывать пользователю в интерфейсе «потому что вы смотрели X»: фильм с максимальным $c_i$ или с максимальным $\Delta_i$? Свяжите ответ с различием **достаточности** и **необходимости** (занятие 08): какая из величин отвечает на вопрос «без X рекомендации бы не было»?

_Ваш ответ:_

---
## Часть 8. Мультимодальность: CLIP (1,2 балла)

CLIP (ViT-B/32) сопоставляет картинку и текст косинусом эмбеддингов. Объяснять можно **обе стороны**: какие патчи картинки и какие слова текста отвечают за сходство.

Картинка — классический пример из COCO: две кошки на розовом пледе и два пульта.

In [ ]:
import requests
from PIL import Image, ImageDraw, ImageFont
from transformers import CLIPModel, CLIPProcessor

CLIP_NAME = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(CLIP_NAME).eval()
clip_proc = CLIPProcessor.from_pretrained(CLIP_NAME)

def _emb(out):
    out = out if torch.is_tensor(out) else out.pooler_output   # transformers v5 возвращает объект
    return out / out.norm(dim=-1, keepdim=True)

@torch.no_grad()
def text_emb(texts):
    t = clip_proc(text=texts, return_tensors="pt", padding=True)
    return _emb(clip_model.get_text_features(**t))

@torch.no_grad()
def image_emb_from_pixels(pv):
    return _emb(clip_model.get_image_features(pixel_values=pv))

def pixels(img):
    return clip_proc(images=[img], return_tensors="pt").pixel_values

CAT_IMG = Image.open(io.BytesIO(requests.get("http://images.cocodataset.org/val2017/000000039769.jpg", timeout=60).content)).convert("RGB")
PV = pixels(CAT_IMG)
PROMPTS = ["a photo of a cat", "a photo of a remote control", "a photo of a dog", "a photo of a pink blanket"]
T_EMB = text_emb(PROMPTS)
CLIP_SHOW = (PV[0] * torch.tensor(clip_proc.image_processor.image_std)[:, None, None]
             + torch.tensor(clip_proc.image_processor.image_mean)[:, None, None]).permute(1, 2, 0).clamp(0, 1).numpy()
plt.figure(figsize=(3.5, 3.5)); plt.imshow(CLIP_SHOW); plt.axis("off"); plt.title("вход CLIP (224×224)"); plt.show()

### 8.1 Zero-shot (0,2 балла)

**[0,2 балла]** Посчитайте косинусы картинки с 4 промптами и вероятности `softmax(100 · cos)`. Почему косинусы такие близкие (≈0.2–0.26), а вероятности — нет?

In [ ]:
IMG_EMB = image_emb_from_pixels(PV)
# CODE PLEASE: косинусы IMG_EMB @ T_EMB.T и softmax(100 * cos)

_Ваш ответ:_

### 8.2 Окклюзия: «сырой» косинус против контрастной цели (0,5 балла)

**a) [0,2 балла]** Реализуйте `clip_occlusion(pv, target_fn, grid=14)`: картинка 224×224 делится на `grid × grid` квадратов, каждый по очереди заменяется нулём (= средний цвет после нормализации), атрибуция квадрата = падение `target_fn(эмбеддинги)`. Цель 1 — **сырой косинус** с каждым из 4 промптов.

**b) [0,2 балла]** Цель 2 — **контрастная**: логарифм вероятности промпта в softmax по 4 промптам (`log_softmax(100·cos)`). Нарисуйте 2×4 карты: строки — цель, столбцы — промпты.

**c) [0,1 балла]** Посчитайте корреляцию Спирмена между картами «cat» и «dog» для обеих целей.

In [ ]:
@torch.no_grad()
def clip_occlusion(pv, target_fn, grid=14, bs=49):
    # CODE PLEASE: grid×grid квадратов зануляются по очереди; атрибуция = target_fn(base) - target_fn(occluded)
    #              верните массив (grid, grid, число_целей)
    pass

raw_target = lambda E: E @ T_EMB.T
contrast_target = lambda E: torch.log_softmax(100 * E @ T_EMB.T, -1)
# CODE PLEASE: OCC_RAW, OCC_CON; сетка карт 2×4 поверх CLIP_SHOW; Spearman(cat, dog) для обеих целей

**Вопрос [без отдельных баллов, входит в 8.4].** Почему карты «сырого» косинуса для cat и dog почти совпадают? Что в эмбеддинге картинки меняется при закрытии любого патча, независимо от промпта?

_Ваш ответ:_

### 8.3 Текстовая сторона: какие слова описания важны (0,3 балла)

**[0,3 балла]** Для подписи `"two cats sleeping on a pink blanket next to two remote controls"` посчитайте leave-one-word-out: важность слова = падение косинуса с картинкой при удалении этого слова. Выведите таблицу. Затем повторите для той же подписи с **другой картинкой** — любой из Imagenette (например, `inette[0][0]`). Какие слова «работают» на конкретную картинку, а какие — на любую?

In [ ]:
CAPTION = "two cats sleeping on a pink blanket next to two remote controls"

def word_loo(caption, img_emb):
    # CODE PLEASE: для каждого слова — подпись без него; важность = cos(полная) - cos(без слова)
    pass

# CODE PLEASE: сравните word_loo для IMG_EMB и для картинки inette[0][0]

### 8.4 Вопрос (0,2 балла)

Свяжите 8.2 и 8.3 с частью 2.4: в каком смысле «сырой косинус» CLIP — аналог карты, не различающей классы? Сформулируйте общее правило выбора **цели** объяснения для моделей, выход которых — сходство, а не логит класса (поиск, ранжирование, RAG-ретриверы).

_Ваш ответ:_

---
## Бонус (1 балл)

### Б.1 Типографская атака на CLIP (0,5 балла)

Goh et al. (2021) показали, что CLIP «читает» текст на картинке: яблоко с наклейкой «iPod» распознаётся как iPod.

**[0,3 балла]** Напишите на картинке с кошками крупными буквами `DOG` (белый прямоугольник + чёрный текст, `PIL.ImageDraw`). Посчитайте вероятности 4 промптов до и после.

**[0,2 балла]** Постройте контрастную окклюзию (8.2) для промпта «dog» на атакованной картинке. Куда смотрит модель? Какой вывод для объяснения мультимодальных моделей: может ли карта атрибуции сама по себе отличить «модель увидела собаку» от «модель прочитала слово»?

In [ ]:
# CODE PLEASE: картинка с надписью DOG, вероятности 4 промптов до/после, контрастная окклюзия для «dog»

_Ваш ответ:_

### Б.2 Итоговая таблица (0,5 балла)

Заполните таблицу для **всех** методов, которые вы применяли в ДЗ (по строке на метод × модальность): единица объяснения, вмешательство, baseline, цель, и **чем проверяли** (истина по построению / согласие методов / ничем). Отметьте, где проверки не было, — и предложите, какой она могла бы быть.

| Модальность | Метод | Единица | Вмешательство | Baseline | Цель | Проверка |
|---|---|---|---|---|---|---|
| таблицы | PDP | признак | подстановка значения | — | прогноз | ... |
| ... | | | | | | |

_Ваш ответ:_